> **QAC reproduction record** — claim `rsa2048-gidney-2025`, reproduction `qsharp-modexp-qre-2026-11`.
>
> This notebook is the executable evidence behind the QAC assessment of Gidney (2025),
> *How to factor 2048 bit RSA integers with less than a million noisy qubits*, arXiv:2505.15917.
> It is kept readable on purpose: the markdown walks through the method so it can be used for learning.
> The **last cell writes `results.json`**, the only source of the numbers the public record attributes to QAC;
> nothing in that file is typed by hand. The environment is pinned in `requirements.lock.txt`,
> and `run.json` records who executed it and when. Re-execute with `qacclaims execute` (about eleven
> minutes on an 11-core laptop).

# RSA-2048 with approximate residue arithmetic: the whole circuit, the real instance

The paper's logical costs come from a reference implementation of its algorithm (Appendix A.1,
`approx_modexp`), and its physical costs from a layout and timing model (Sec. 3.2). This notebook
rebuilds both for the actual RSA-2048 challenge number:

1. **The circuit.** `modexp.qs` implements `approx_modexp` operation for operation as one Q# circuit,
   with the lookup of Appendix A.4 and the phaseup of Appendix A.3 (`lookup_phaseup.qs`). It is checked
   by simulation on small instances.
2. **The instance.** `rns_precompute.py` finds a real residue number system for RSA-2048 (Eq. 17 and
   Assumption 1) and builds every lookup table of the real circuit. It is checked against the
   precomputation in the paper's published code on the small instances.
3. **The count.** The circuit is counted at full size on the real primes, and corrected to the real
   tables lookup by lookup, with the correction itself checked against Q#.
4. **The failed-shot rate.** Recomputed with the paper's bound for the real residue system, and
   measured by running the approximate arithmetic on 10,000 random exponents.
5. **The physical cost.** The layout and timing model of Sec. 3.2 (`physical_layout.py`) applied to the
   circuit's own counts, and, as an independent check, the QDK resource estimator.

## Parameters

The RSA-2048 row of Table 5 (p. 17) fixes the algorithm's parameters, and the abstract (p. 1) and
Section 3.2 (pp. 18–20) fix the physical assumptions. The generator g = 3 is the one in the paper's
own RSA-2048 test configuration; any element of the group works the same way.

In [1]:
import math
import rns_precompute as rp
import physical_layout as layout_model

# Table 5, n = 2048 row (p. 17)
N_BITS = 2048
S = 8                  # Ekerå–Håstad parameter
L = 21                 # ℓ, bit length of the primes in the residue number system
W1, W3, W4 = 6, 3, 5   # window sizes of loop1, loop3/unloop3, loop4
F = 33                 # f, length of the truncated accumulator
M = 1280               # m, number of input qubits
P_DEVIANT_REPORTED = 0.0125

PARAMS = rp.TABLE5_RSA2048
assert (PARAMS.modulus.bit_length(), PARAMS.num_input_qubits, PARAMS.prime_bit_length) == (N_BITS, M, L)
assert (PARAMS.window1, PARAMS.window3a, PARAMS.window3b, PARAMS.window4, PARAMS.len_accumulator) == (W1, W3, W3, W4, F)
assert M == math.ceil(N_BITS / 2 + N_BITS / S)

# Physical assumptions (abstract, p. 1; Sec. 3.2, pp. 18-20)
PHYSICAL_ERROR_RATE = layout_model.PHYSICAL_ERROR_RATE
CYCLE_TIME_NS = int(layout_model.CYCLE_TIME_US * 1000)
PER_SHOT_ERROR_BUDGET = round(1 - 0.933, 3)   # Sec. 3.2: a shot has a 93.3% chance of no logical error

LEN_DLOG = PARAMS.len_dlog
NUM_WINDOWS_1 = PARAMS.num_windows1
print(f"len dlog = {LEN_DLOG}, windows: {NUM_WINDOWS_1} / {PARAMS.num_windows3a} / {PARAMS.num_windows4}, "
      f"mask bits = {PARAMS.mask_bits}, wraparound gap = {PARAMS.min_wraparound_gap}")

len dlog = 32, windows: 214 / 7 / 5, mask bits = 26, wraparound gap = 33


## The circuit

The Q# files sit next to this notebook. The cell loads them into the QDK.

In [2]:
import json
from pathlib import Path
import numpy as np
import qdk
from qdk import qsharp

qsharp.init()
for qs_file in ("lookup_phaseup.qs", "modexp.qs", "lookup_cost.qs"):
    qsharp.eval(Path(qs_file).read_text())

def toffolis(counts):
    return counts["cczCount"] + counts["ccixCount"]

def make_cfg(d, compare_mode=0):
    return qdk.code.MakeConfig(
        d["periods"], d["len_dlog"], d["len_acc"], d["mask_bits"], d["trunc"],
        d["w1"], d["w3a"], d["w3b"], d["w4"], d["num_input_qubits"],
        d["table1"], d["table3a"], d["table3b"], d["table3c"], d["table4"],
        False, compare_mode, False)

def rsa_cfg(periods, tables, synthetic, compare_mode, cache, num_input_qubits=M):
    t1, t3a, t3b, t3c, t4 = tables if tables else ([], [], [], [], [])
    return qdk.code.MakeConfig(list(map(int, periods)), LEN_DLOG, F, PARAMS.mask_bits, PARAMS.trunc,
                               W1, W3, W3, W4, num_input_qubits, t1, t3a, t3b, t3c, t4,
                               synthetic, compare_mode, cache)

## Verifying the circuit on small instances

The files in `test_vectors/` hold small problem instances, from 32-bit to 100-bit moduli, including one
with the RSA-2048 window sizes. Each holds a residue number system and the lookup tables built with
the precomputation in the paper's published code
([Zenodo](https://doi.org/10.5281/zenodo.15347487), CC BY 4.0), and the output the paper's own
Appendix A.1 code gives for several exponents. The checks are:

1. **Values.** For each test exponent, the circuit's output register equals the expected value, the
   exponent register is returned unchanged, and every temporary qubit is released in $|0\rangle$ (the
   simulator stops with an error otherwise).
2. **Phases.** On the two smallest instances, the exponent and the mask are put in superposition. A
   correct circuit, including every deferred phase correction, ends in exactly the uniform
   superposition of $|e\rangle|(s + V(e)) \bmod t\rangle$, which is checked amplitude by amplitude.
   A deliberately broken variant that skips the measurement-triggered comparisons must fail the
   check, which shows the check can catch missing phase corrections.

In [3]:
import time

def check_basis(d):
    cfg = make_cfg(d)
    ok = True
    for c in d["cases"]:
        e_bits = [bool((c["e"] >> b) & 1) for b in range(d["num_input_qubits"])]
        [(result, e_out_bits)] = qsharp.run(qdk.code.RunBasis, 1, cfg, e_bits, c["s"])
        e_out = sum(1 << b for b, v in enumerate(e_out_bits) if v)
        ok &= result == c["author_result"] == c["model_result"] and e_out == c["e"]
    return ok

def expected_superposed_state(d):
    m, f1, mb, trunc = d["num_input_qubits"], d["len_acc"] + 1, d["mask_bits"], d["trunc"]
    n, amp, state = m + f1, (2 ** -(m + mb)) ** 0.5, {}
    for e in range(1 << m):
        for s in range(1 << mb):
            acc = (s + d["model_all"][e]) % trunc
            bits = [(e >> b) & 1 for b in range(m)] + [(acc >> b) & 1 for b in range(f1)]
            state[sum(bit << (n - 1 - k) for k, bit in enumerate(bits))] = amp
    return state

def check_superposed(d, shots=3):
    expected = expected_superposed_state(d)
    runs = qsharp.run(qdk.code.RunSuperposed, shots, make_cfg(d), d["num_input_qubits"], save_events=True)
    ok = all(r["dumps"] and r["dumps"][0].check_eq(expected) for r in runs)
    broken = qsharp.run(qdk.code.RunSuperposed, 2, make_cfg(d, compare_mode=2), d["num_input_qubits"], save_events=True)
    caught = all((not r["dumps"]) or (not r["dumps"][0].check_eq(expected)) for r in broken)
    return ok, caught

TEST_VECTORS = {path.stem: json.loads(path.read_text()) for path in sorted(Path("test_vectors").glob("*.json"))}
VERIFICATION = {}
for name, d in TEST_VECTORS.items():
    t0 = time.time()
    entry = {"modulus_bits": d["modulus"].bit_length(), "input_qubits": d["num_input_qubits"],
             "primes": len(d["periods"]), "basis_cases": len(d["cases"]), "values_correct": check_basis(d)}
    if d.get("superposed"):
        entry["phases_correct"], entry["broken_variant_caught"] = check_superposed(d)
    VERIFICATION[name] = entry
    print(f"{name}: {entry} ({time.time() - t0:.0f}s)")

assert all(all(v for k, v in e.items() if k in ("values_correct", "phases_correct", "broken_variant_caught"))
           for e in VERIFICATION.values()), "verification failed"

rsa_windows_100bit: {'modulus_bits': 100, 'input_qubits': 20, 'primes': 34, 'basis_cases': 6, 'values_correct': True} (25s)


tiny_rsa_windows: {'modulus_bits': 32, 'input_qubits': 8, 'primes': 8, 'basis_cases': 12, 'values_correct': True, 'phases_correct': True, 'broken_variant_caught': True} (28s)


tiny_w2: {'modulus_bits': 32, 'input_qubits': 4, 'primes': 8, 'basis_cases': 16, 'values_correct': True, 'phases_correct': True, 'broken_variant_caught': True} (14s)


## Checking the precomputation on the same instances

`rns_precompute.py` is an independent implementation of the paper's classical precomputation: the
search for a residue number system (Eq. 17), the per-prime generators, the discrete logs of table1, the
small exponentiation tables of loop3, and the truncated CRT contributions of loop4 (Eq. 21). On every
test instance, given the instance's primes, it must rebuild the generators and all five tables exactly;
its model of the approximate output must equal the paper's reference implementation; and its own
residue-system search, run on the instance's parameters, must return a system meeting every condition.

In [4]:
WRAPAROUND_GAPS = {"tiny_w2": 12, "tiny_rsa_windows": 12, "rsa_windows_100bit": 20}
PRECOMPUTE_CHECKS = {}
for name, d in TEST_VECTORS.items():
    small = rp.params_from_vector(d, WRAPAROUND_GAPS[name])
    tables = rp.build_tables(small, d["periods"])
    checks = {"generators": tables["generators"] == d["generators"],
              **{k: tables[k] == d[k] for k in ("table1", "table3a", "table3b", "table3c", "table4")},
              "model": all(rp.model_output(small, d["periods"], d["table4"], c["e"], c["s"]) == c["author_result"]
                           for c in d["cases"])}
    rp.verify_residue_system(small, rp.find_residue_system(small, seed=0, workers=1).periods)  # raises if invalid
    checks["own_search_valid"] = True
    PRECOMPUTE_CHECKS[name] = checks
    print(name, checks)
assert all(all(c.values()) for c in PRECOMPUTE_CHECKS.values()), "precomputation check failed"

rsa_windows_100bit {'generators': True, 'table1': True, 'table3a': True, 'table3b': True, 'table3c': True, 'table4': True, 'model': True, 'own_search_valid': True}
tiny_rsa_windows {'generators': True, 'table1': True, 'table3a': True, 'table3b': True, 'table3c': True, 'table4': True, 'model': True, 'own_search_valid': True}
tiny_w2 {'generators': True, 'table1': True, 'table3a': True, 'table3b': True, 'table3c': True, 'table4': True, 'model': True, 'own_search_valid': True}


## The real residue number system for RSA-2048

The circuit computes $g^e \bmod N$ modulo many small primes. Their product $L$ must be at least
$N^{\lceil m/w_1 \rceil}$, and, so that missed wraparounds modulo $L$ cost nothing, $L \bmod N$ must have a
modular deviation below $2^{-f}$ (Eq. 17). The search starts from the largest 21-bit primes that give
enough capacity, adds pairs of further primes until $L \bmod N$ shares two primes with the set, and
divides those out, which makes $L \bmod N$ tiny (the approach of the paper's published code, done
here incrementally). Primes that divide one of table1's multipliers are excluded, since their
discrete logs would not exist.

In [5]:
t0 = time.time()
RESIDUE_SYSTEM = rp.find_residue_system(PARAMS, seed=0)
PERIODS = np.array(RESIDUE_SYSTEM.periods, dtype=np.int64)
RNS = rp.verify_residue_system(PARAMS, RESIDUE_SYSTEM.periods)   # raises if any condition fails
NUM_PRIMES = RNS["num_primes"]
print(f"{NUM_PRIMES:,} primes from {PERIODS[0]:,} to {PERIODS[-1]:,} ({time.time() - t0:.0f}s)")
print(f"L has {RNS['L_bits']:,} bits (needs {RNS['capacity_bits_needed']:,}); "
      f"deviation of L mod N = 2^{RNS['L_mod_N_deviation_log2']:.1f} (needs < 2^{RNS['threshold_log2']})")
print(f"excluded primes: {len(RESIDUE_SYSTEM.excluded_primes)}; candidate pairs tried: {RESIDUE_SYSTEM.candidates_tried}")

20,979 primes from 1,783,163 to 2,097,143 (64s)
L has 438,230 bits (needs 438,196); deviation of L mod N = 2^-47.7 (needs < 2^-33)
excluded primes: 668; candidate pairs tried: 1837


## Every lookup table of the real circuit

The scan builds all of the circuit's tables for these primes (table1's discrete logs alone hold about
290 million entries) one chunk of primes at a time, and computes the Toffoli count of every lookup the
circuit performs on them. The optimized lookup's cost is fixed by its address size, except that a pair
of power products whose output mask is all zeros needs no Toffoli. Real tables have a few such pairs;
pseudo-random tables have essentially none.

The same pass also evaluates loop4's truncated contributions for 10,000 random exponents, used below
for the failed-shot rate.

In [6]:
t0 = time.time()
SAMPLE_EXPONENTS, SAMPLE_DIGITS = rp.sample_exponents(PARAMS, 10000, 2025)
SCAN = rp.scan(PARAMS, PERIODS, digits=SAMPLE_DIGITS)
print(f"scanned {SCAN.num_primes:,} primes in {time.time() - t0:.0f}s; table digest {SCAN.table_digest[:16]}…")
for kind in rp.KINDS:
    print(f"  {kind:12s} Toffolis {int(SCAN.lookup_toffolis[kind].sum()):>13,}   zero-mask pairs {int(SCAN.zero_pairs[kind].sum()):>8,}")

scanned 20,979 primes in 202s; table digest 88919aee03944d6f…
  loop1        Toffolis   254,568,573   zero-mask pairs  158,445
  loop3        Toffolis    43,033,292   zero-mask pairs   15,616
  loop4        Toffolis     2,202,651   zero-mask pairs      144
  loop4_cmp    Toffolis     2,202,753   zero-mask pairs       42
  unloop3      Toffolis    83,634,361   zero-mask pairs   71,849
  unloop3_cmp  Toffolis    83,634,761   zero-mask pairs   71,449


### Checking the lookup costs against Q#

Two checks tie the classical cost function to the circuit. First, for every lookup the circuit performs
on 100 of the real primes (tens of thousands of real tables, every kind of lookup), the Q# count of the
lookup equals the cost function. Second, for three real primes, the
whole iteration of the circuit for that prime, built with its real tables and counted without caching,
equals the pseudo-random count plus the per-lookup correction, in both comparison modes.

In [7]:
import random

def bool_rows(values, width):
    return [[bool((int(v) >> b) & 1) for b in range(width)] for v in values]

COMPARISON_KINDS = ("loop4_cmp", "unloop3_cmp")
def mode_sum(costs, mode):
    return sum(v for k, v in costs.items() if mode == 1 or k not in COMPARISON_KINDS)

t0 = time.time()
rng = random.Random(7)
sample_idx = sorted(set(rng.sample(range(NUM_PRIMES), 99)) | {NUM_PRIMES - 1})
sample_tables = rp.prime_tables(PARAMS, PERIODS, sample_idx)
groups = {}
for pt in sample_tables:
    T = {k: np.asarray(pt[k])[None] for k in ("t1", "t3a", "t3b", "t3c", "t4")}
    period = np.array([pt["period"]], dtype=np.int64)
    for spec in rp.lookup_specs(PARAMS):
        groups.setdefault(f"{spec.kind}:{spec.table}:n{spec.n}:w{spec.width}", []).append(
            (spec, rp.lookup_data(spec, T, period, PARAMS)[0]))
TABLE_CHECKS = {}
for key, items in sorted(groups.items()):
    agree = sum(int(rp.lookup_toffolis(data[None], spec.n)[0][0]) ==
                toffolis(qsharp.logical_counts(qdk.code.LookupCost, bool_rows(data, spec.width), spec.n, spec.width))
                for spec, data in items)
    TABLE_CHECKS[key] = (agree, len(items))
print({k: f"{a}/{n}" for k, (a, n) in TABLE_CHECKS.items()}, f"({time.time() - t0:.0f}s)")
assert all(a == n for a, n in TABLE_CHECKS.values())

t0 = time.time()
PRIME_CHECKS = []
for pt in [p for p in sample_tables if p["index"] in (sample_idx[0], sample_idx[len(sample_idx) // 2], NUM_PRIMES - 1)]:
    p = pt["period"]
    T = {k: np.asarray(pt[k])[None] for k in ("t1", "t3a", "t3b", "t3c", "t4")}
    real_iter = {k: int(v[0]) for k, v in rp.iteration_costs(PARAMS, np.array([p], np.int64), T)["toffolis"].items()}
    real_final = rp.final_loop1_costs(PARAMS, np.asarray(pt["t1_next"]))[0]
    synth_iter, synth_final = rp.synthetic_iteration_costs(PARAMS, 0, p), rp.synthetic_final_cost(PARAMS, 1)
    tables = ([np.asarray(pt["t1"]).tolist(), np.asarray(pt["t1_next"]).tolist()], [np.asarray(pt["t3a"]).tolist()],
              [np.asarray(pt["t3b"]).tolist()], [np.asarray(pt["t3c"]).tolist()], [np.asarray(pt["t4"]).tolist()])
    for mode in (1, 2):
        real = qsharp.logical_counts(qdk.code.CountFull, rsa_cfg([p], tables, False, mode, False), M)
        synth = qsharp.logical_counts(qdk.code.CountFull, rsa_cfg([p], None, True, mode, False), M)
        predicted = toffolis(synth) + mode_sum(real_iter, mode) - mode_sum(synth_iter, mode) + real_final - synth_final
        PRIME_CHECKS.append((pt["index"], mode, toffolis(real), predicted))
print("one-prime iterations with real tables (index, mode, Q#, predicted):", PRIME_CHECKS, f"({time.time() - t0:.0f}s)")
assert all(real == predicted for _, _, real, predicted in PRIME_CHECKS)

{'loop1:t1:n2:w32': '100/100', 'loop1:t1:n6:w32': '21300/21300', 'loop3:t3a:n6:w22': '3500/3500', 'loop3:t3c:n6:w22': '100/100', 'loop4:t4:n2:w34': '100/100', 'loop4:t4:n5:w34': '400/400', 'loop4_cmp:t4:n2:w33': '100/100', 'loop4_cmp:t4:n5:w33': '400/400', 'unloop3:t3a:n6:w22': '3500/3500', 'unloop3:t3b:n6:w22': '3500/3500', 'unloop3_cmp:t3a:n6:w21': '3500/3500', 'unloop3_cmp:t3b:n6:w21': '3500/3500'} (209s)


one-prime iterations with real tables (index, mode, Q#, predicted): [(1228, 1, 57330, 57330), (1228, 2, 51681, 51681), (11848, 1, 57336, 57336), (11848, 2, 51682, 51682), (20978, 1, 57351, 57351), (20978, 2, 51712, 51712)] (42s)


## Counting the whole circuit

The same `ApproxModExp` operation is counted on all of the real primes. Every prime's iteration has the
same structure, so each is wrapped in the QDK's estimate caching: traced once and accounted for |P|
times, with pseudo-random tables; the cell checks on a smaller instance that caching changes nothing.
The scan's per-lookup correction then turns that count into the count with the real tables.

In `loop4` and `unloop3`, a comparison runs only when an X-basis measurement returns 1, which happens
with probability exactly ½, so the expected count is the mean of a count where these comparisons
always run (mode 1) and one where they never do (mode 2). The circuit's peak qubit count comes directly
from Q#.

As a cross-check of the operation tallies used below for the runtime, the cell also counts the
circuit's additions, lookups, and phaseups per subroutine and checks that their Toffoli costs add up
to the Q# count.

In [8]:
def count(periods, compare_mode, cache, num_input_qubits=M):
    return qsharp.logical_counts(qdk.code.CountFull, rsa_cfg(periods, None, True, compare_mode, cache, num_input_qubits),
                                 num_input_qubits)

medium = PERIODS[-6:]
for mode in (1, 2):
    assert count(medium, mode, False, 60) == count(medium, mode, True, 60)
print("caching check (60 input qubits, 6 primes): identical counts")

synth_iteration = rp.synthetic_iteration_costs(PARAMS, 0, int(PERIODS[0]))
synth_final = rp.synthetic_final_cost(PARAMS, NUM_PRIMES)
real_lookups = {k: int(SCAN.lookup_toffolis[k].sum()) for k in rp.KINDS}
FULL = {}
for mode in (1, 2):
    c = count(PERIODS, mode, True)
    correction = (mode_sum(real_lookups, mode) - NUM_PRIMES * mode_sum(synth_iteration, mode)
                  + SCAN.final_loop1_toffolis - synth_final)
    FULL[mode] = dict(pseudo_random_tables=toffolis(c), correction=correction, real=toffolis(c) + correction,
                      qubits=c["numQubits"])
    print(f"mode {mode}: {FULL[mode]}")

TOFFOLIS_PER_SHOT = round((FULL[1]["real"] + FULL[2]["real"]) / 2)
LOGICAL_QUBITS = max(FULL[1]["qubits"], FULL[2]["qubits"])
HOT_LOGICAL_QUBITS = LOGICAL_QUBITS - M

# Operation tallies per shot (Appendix A.1 / Table 3, with the measured comparisons at probability ½).
def windows(total, w):
    return [min(w, total - j * w) for j in range(-(-total // w))]
def tof_add(n): return n - 1
def tof_lookup(a): return (1 << a) - a - 1
def tof_phaseup(a): return max(0, (1 << (a // 2)) - a // 2 - 1) + max(0, (1 << (a - a // 2)) - (a - a // 2) - 1)

def tally(compare):
    w1s, w4s = windows(M, W1), windows(L + 1, W4)
    w3 = -(-L // W3)
    inner, a3 = max(0, w3 - 2) * w3, 2 * W3
    steps = range(L, LEN_DLOG)
    per = {  # (additions, lookups, phaseups, Toffolis) per call, and calls per shot
        "loop1": (len(w1s), len(w1s), 0, sum(tof_lookup(a) + tof_add(LEN_DLOG) for a in w1s), NUM_PRIMES),
        "loop2": (2 * len(steps), 0, 0, sum(tof_add(k + 1) + tof_add(k) for k in steps), NUM_PRIMES),
        "loop3": (2 * inner, 1 + inner, 0, tof_lookup(a3) + inner * (tof_lookup(a3) + tof_add(L + 1) + tof_add(L)), NUM_PRIMES),
        "loop4": ((2 + compare) * len(w4s), (1 + compare) * len(w4s), len(w4s),
                  sum(tof_lookup(a) + tof_add(F + 1) + tof_add(F) + compare * (tof_lookup(a) + tof_add(F)) + tof_phaseup(a)
                      for a in w4s), NUM_PRIMES),
        "unloop3": ((4 + 2 * compare) * inner, (2 + 2 * compare) * inner, 2 * inner + 1,
                    2 * inner * (tof_lookup(a3) + tof_add(L + 1) + tof_add(L) + compare * (tof_lookup(a3) + tof_add(L))
                                 + tof_phaseup(a3)) + tof_phaseup(a3), NUM_PRIMES),
        "final": (len(w1s), len(w1s), len(w1s), sum(tof_lookup(a) + tof_add(LEN_DLOG) + tof_phaseup(a) for a in w1s), 1),
    }
    per["unloop2"] = per["loop2"]
    totals = [sum(v[i] * v[4] for v in per.values()) for i in range(4)]
    return dict(additions=totals[0], lookups=totals[1], phaseups=totals[2], toffolis=totals[3])

for mode, compare in ((1, 1), (2, 0)):
    assert tally(compare)["toffolis"] == FULL[mode]["pseudo_random_tables"], "operation tally does not match the circuit"
OPS_PER_SHOT = tally(0.5)
print(f"Toffolis per shot: mode 1 {FULL[1]['real']:,}, mode 2 {FULL[2]['real']:,}, expected {TOFFOLIS_PER_SHOT:,}")
print(f"operations per shot: {OPS_PER_SHOT['additions']:,.1f} additions, {OPS_PER_SHOT['lookups']:,.1f} lookups, "
      f"{OPS_PER_SHOT['phaseups']:,} phaseups")
print(f"peak logical qubits: {LOGICAL_QUBITS}  (Table 5: 1399); outside the input register: {HOT_LOGICAL_QUBITS}")

caching check (60 input qubits, 6 primes): identical counts


mode 1: {'pseudo_random_tables': 774523202, 'correction': -317545, 'real': 774205657, 'qubits': 1436}


mode 2: {'pseudo_random_tables': 655886957, 'correction': -246054, 'real': 655640903, 'qubits': 1436}
Toffolis per shot: mode 1 774,205,657, mode 2 655,640,903, expected 714,923,280
operations per shot: 10,814,888.5 additions, 7,605,101.5 lookups, 1,594,618 phaseups
peak logical qubits: 1436  (Table 5: 1399); outside the input register: 156


## The failed-shot rate

A shot fails with probability $P_\text{deviant} \le S + \epsilon/S$ (Eq. 43), where $S$ is the fraction
of the output space the superposition mask covers and $\epsilon$ bounds the modular deviation of the
approximate result: each truncated addition contributes at most $2 \cdot 2^{-f}$ from dropped carries
and the residue system's wraparound contributes at most $2^{-\text{gap}}$ per addition. Table 5's
1.25% is this bound. Here it is recomputed for the real residue system.

The bound is a worst case. To see how the approximation actually behaves, the cell takes 10,000 random
exponents, computes the circuit's approximate output with the real loop4 tables (no mask), and compares
it with the exact $g^e \bmod N$ shifted to $f$ bits.

In [9]:
BOUND = rp.p_deviant_bound(PARAMS, NUM_PRIMES)
BOUND_ACTUAL_L = rp.p_deviant_bound(PARAMS, NUM_PRIMES, wraparound_deviation=RNS["L_mod_N_deviation"])
t0 = time.time()
DEVIATIONS = rp.deviations(PARAMS, rp.exact_values(PARAMS, SAMPLE_EXPONENTS), SCAN.table4_sum)
MASK_PROPORTION = BOUND["mask_proportion"]
P_DEVIANT = BOUND["p_deviant"]
SHOTS = (S + 1) / (1 - P_DEVIANT)
print(f"bound with the real residue system: S = 2^{math.log2(MASK_PROPORTION):.0f}, epsilon = {BOUND['epsilon']:.3e}, "
      f"P_deviant = {P_DEVIANT:.4%}  (Table 5: {P_DEVIANT_REPORTED:.2%})")
print(f"bound with the actual deviation of L mod N: {BOUND_ACTUAL_L['p_deviant']:.4%}")
print(f"measured deviation over {len(DEVIATIONS):,} exponents: max {DEVIATIONS.max():.3e} "
      f"({DEVIATIONS.max() / BOUND['epsilon']:.1%} of epsilon), mean {DEVIATIONS.mean():.3e}, "
      f"spread {DEVIATIONS.max() - DEVIATIONS.min():.1e} ({time.time() - t0:.0f}s)")
print(f"S + mean deviation / S = {MASK_PROPORTION + DEVIATIONS.mean() / MASK_PROPORTION:.3%}")
assert DEVIATIONS.max() <= BOUND["epsilon"]

EXPECTED_TOFFOLIS = round(TOFFOLIS_PER_SHOT * SHOTS)
print(f"expected shots (s + 1)/(1 - P_deviant) = {SHOTS:.4f}; expected Toffolis per factoring {EXPECTED_TOFFOLIS:,}  (Table 5: 6.5e+09)")

bound with the real residue system: S = 2^-7, epsilon = 3.663e-05, P_deviant = 1.2502%  (Table 5: 1.25%)
bound with the actual deviation of L mod N: 1.0939%
measured deviation over 10,000 exponents: max 1.910e-06 (5.2% of epsilon), mean 1.853e-06, spread 1.2e-07 (19s)
S + mean deviation / S = 0.805%
expected shots (s + 1)/(1 - P_deviant) = 9.1139; expected Toffolis per factoring 6,515,767,508  (Table 5: 6.5e+09)


## Physical cost: the paper's layout, rebuilt

`physical_layout.py` implements Section 3.2 as code, with each of the paper's constants and its
locator:

- **Code distance.** Hot patches use the smallest distance whose Fig. 6 fit,
  $p_L \approx 3.5^{-d}/40$ per patch per round, reaches the target of $10^{-15}$; each takes
  $2(d+1)^2$ physical qubits. The input register sits in cold storage, surface codes yoked by a 2D
  parity check code, at the 430 physical qubits per logical qubit Fig. 6 gives at that target.
- **Factories.** Six factories, each 3×4 hot patches, make CCZ states with magic state cultivation
  (30,000 qubit-rounds per T state, from [GSJ24] Fig. 1) and 8T-to-CCZ distillation (6 layers of
  temporally encoded lattice surgery). The model checks that this fits in the paper's 150 rounds per
  CCZ state, which with six factories is one CCZ state per 25 µs.
- **Operation durations.** The model checks the paper's bounds of 2 ms per addition, 2 ms per lookup,
  and 1 ms per phaseup against the largest operation of each kind in this circuit, then multiplies
  them by the circuit's operation counts per shot.
- **Space.** Cold storage for the input register, hot storage for every other qubit at the circuit's
  peak, and a 7×18 compute region of factories and lattice-surgery workspace.
- **Logical errors.** Every logical patch is protected at $10^{-15}$ per round, which gives the chance
  that a shot has no logical error; the expected runtime divides by it.

In [10]:
LAYOUT = layout_model.layout(
    input_qubits=M, peak_logical_qubits=LOGICAL_QUBITS,
    additions_per_shot=OPS_PER_SHOT["additions"], lookups_per_shot=OPS_PER_SHOT["lookups"],
    phaseups_per_shot=OPS_PER_SHOT["phaseups"], shots=SHOTS,
    largest_adder_toffolis=tof_add(max(LEN_DLOG, F + 1, L + 1)),
    largest_lookup_address_bits=max(W1, 2 * W3, W4), largest_phaseup_toffolis=tof_phaseup(2 * W3))
for key in ("code_distance", "ccz_rounds", "adder_bound_ms", "lookup_bound_ms", "phaseup_bound_ms",
            "cold_physical", "hot_physical", "factory_physical", "workspace_physical", "physical_qubits",
            "logical_patches", "p_no_logical_error"):
    print(f"{key:20s} {LAYOUT[key]:,.4g}" if isinstance(LAYOUT[key], float) else f"{key:20s} {LAYOUT[key]:,}")
print(f"per shot {LAYOUT['seconds_per_shot'] / 3600:.2f} h (paper: 12.07 h); "
      f"per factoring {LAYOUT['runtime_seconds'] / 86400:.2f} days (paper: 4.96 days)")

code_distance        25
ccz_rounds           114.8
adder_bound_ms       1.65
lookup_bound_ms      1.575
phaseup_bound_ms     0.4
cold_physical        550,400
hot_physical         210,912
factory_physical     97,344
workspace_physical   73,008
physical_qubits      931,664
logical_patches      1,562
p_no_logical_error   0.9417
per shot 10.68 h (paper: 12.07 h); per factoring 4.31 days (paper: 4.96 days)


## Physical cost: an independent check with the QDK resource estimator

The estimator works on a *trace*: the operations of one shot and the qubits they act on. We hand it
the circuit's counts:

- one shot is `TOFFOLIS_PER_SHOT` Toffoli gates, executed one after another;
- the input register (m qubits) is idle for most of the shot, so it is declared as **memory**;
- the remaining qubits at the circuit's peak are **compute** qubits.

The physical model follows the paper's assumptions where the estimator can express them: a
gate-based architecture with a 0.1% error rate, surface codes with a 1 µs cycle, Toffoli states
from magic state cultivation (`GSJ24Factory`) followed by the 8T-to-CCX factory of Fig. 24 of the
same paper (`GSJ24CCXFactory`), and Toffolis consumed as magic states through lattice surgery
(`PSSPC` with `ccx_magic_states=True`). The error budget per shot is the paper's implied 6.7%. The
estimator chooses the code distances and the number of factories itself, and uses its own timing.

Two scenarios: **`qre-yoked-cultivation`** stores the memory qubits in surface codes yoked by 2D
parity checks (`TwoDimensionalYokedSurfaceCode`), as the paper's cold storage does;
**`qre-hot-storage-only`** keeps every qubit in ordinary surface-code patches, to show how much of the
claim rests on yoked storage.

In [11]:
import re
from qdk.qre import Application, Trace, PSSPC, LatticeSurgery, estimate, instruction_ids
from qdk.qre.models import (GateBased, SurfaceCode, GSJ24Factory, GSJ24CCXFactory,
                            TwoDimensionalYokedSurfaceCode)

class FactoringShot(Application[None]):
    # One shot of the circuit: TOFFOLIS_PER_SHOT sequential Toffolis, input register optionally in memory.
    def __init__(self, yoked_memory: bool):
        self.yoked_memory = yoked_memory

    def get_trace(self, parameters: None) -> Trace:
        trace = Trace(HOT_LOGICAL_QUBITS if self.yoked_memory else LOGICAL_QUBITS)
        block = trace.add_block(repetitions=TOFFOLIS_PER_SHOT)
        block.add_operation(instruction_ids.CCX, [0, 1, 2])
        if self.yoked_memory:
            trace.increment_memory_qubits(M)
        return trace

ARCHITECTURE_PARAMS = dict(error_rate=PHYSICAL_ERROR_RATE, gate_time=100, measurement_time=500)  # ns
architecture = GateBased(**ARCHITECTURE_PARAMS)
surface_code = SurfaceCode.q(code_cycle_override=CYCLE_TIME_NS)
toffoli_factories = GSJ24Factory.q() * GSJ24CCXFactory.q()
trace_query = PSSPC.q(ccx_magic_states=True) * LatticeSurgery.q()

scenarios = {
    "qre-yoked-cultivation": estimate(
        FactoringShot(yoked_memory=True), architecture,
        surface_code * toffoli_factories * TwoDimensionalYokedSurfaceCode.q(source=SurfaceCode.q(code_cycle_override=CYCLE_TIME_NS)),
        trace_query, max_error=PER_SHOT_ERROR_BUDGET, name="yoked storage"),
    "qre-hot-storage-only": estimate(
        FactoringShot(yoked_memory=False), architecture,
        surface_code * toffoli_factories,
        trace_query, max_error=PER_SHOT_ERROR_BUDGET, name="hot storage only"),
}

def compute_distance(entry):
    # The code distance of the compute region: the top-level lattice-surgery surface code in the estimate's source.
    match = re.search(r"^LATTICE_SURGERY @ SurfaceCode\(.*distance=(\d+)\)", str(entry.source), re.MULTILINE)
    return int(match.group(1))

for scenario_id, table in scenarios.items():
    table.add_factory_summary_column()
    table.add_qubit_partition_column()
    display(table.as_frame())
    print(f"{scenario_id}: compute-region code distance {compute_distance(table[0])}")

,name,qubits,runtime,error,factories,physical_compute_qubits,physical_factory_qubits,physical_memory_qubits
0,yoked storage,734185,0 days 13:42:09.706320,0.047422,2×CCX,368893,21144,344148


qre-yoked-cultivation: compute-region code distance 23


,name,qubits,runtime,error,factories,physical_compute_qubits,physical_factory_qubits,physical_memory_qubits
0,hot storage only,3744413,0 days 14:53:39.246000,0.044049,2×CCX,3723269,21144,0


qre-hot-storage-only: compute-region code distance 25


## Results

The rebuilt layout of Section 3.2 is the headline physical scenario, since it is the paper's own model
applied to the circuit's own counts. The estimator's scenarios follow it. For the estimator, as in
Section 3.2, the expected runtime is the per-shot runtime times the shots divided by the probability of
no logical error.

In [12]:
import platform
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata

CLAIM_ID = "rsa2048-gidney-2025"
REPRODUCTION_ID = "qsharp-modexp-qre-2026-11"

def layout_record():
    return {
        "scenario_id": "paper-layout-rebuilt",
        "architecture": {
            "physical_error_rate": PHYSICAL_ERROR_RATE,
            "gate_time_ns": ARCHITECTURE_PARAMS["gate_time"],
            "measurement_time_ns": ARCHITECTURE_PARAMS["measurement_time"],
        },
        "code": {"family": "surface", "distance": LAYOUT["code_distance"], "distance_source": "selected_by_tool",
                 "cycle_time_ns": CYCLE_TIME_NS},
        "factories": f"{layout_model.FACTORY_COUNT} factories of magic state cultivation followed by 8T-to-CCZ "
                     f"distillation, {layout_model.CCZ_ROUNDS_WITH_SLACK} rounds per CCZ state (Sec. 3.2)",
        "synthesis": "Toffolis consumed as CCZ states; operation durations bounded at 2 ms per addition, "
                     "2 ms per lookup, and 1 ms per phaseup (Sec. 3.2); input register in 2D-yoked cold storage",
        "physical_qubits": int(LAYOUT["physical_qubits"]),
        "compute_qubits": int(LAYOUT["hot_physical"] + LAYOUT["workspace_physical"]),
        "factory_qubits": int(LAYOUT["factory_physical"]),
        "memory_qubits": int(LAYOUT["cold_physical"]),
        "runtime_seconds": LAYOUT["runtime_seconds"],
        "runtime_seconds_per_shot": LAYOUT["seconds_per_shot"],
        "error_budget": PER_SHOT_ERROR_BUDGET,
        "achieved_error": 1 - LAYOUT["p_no_logical_error"],
    }

def qre_record(scenario_id, table, description):
    entry = table[0]
    row = table.as_frame().iloc[0]
    per_shot_seconds = row["runtime"].total_seconds()
    return {
        "scenario_id": scenario_id,
        "architecture": {
            "physical_error_rate": ARCHITECTURE_PARAMS["error_rate"],
            "gate_time_ns": ARCHITECTURE_PARAMS["gate_time"],
            "measurement_time_ns": ARCHITECTURE_PARAMS["measurement_time"],
        },
        "code": {"family": "surface", "distance": compute_distance(entry), "distance_source": "selected_by_tool",
                 "cycle_time_ns": CYCLE_TIME_NS},
        "factories": f"{row['factories']} (8T-to-CCX, GSJ24 Fig. 24) fed by magic state cultivation (GSJ24)",
        "synthesis": f"Toffolis consumed as CCX magic states through lattice surgery (PSSPC); {description}",
        "physical_qubits": int(row["qubits"]),
        "compute_qubits": int(row["physical_compute_qubits"]),
        "factory_qubits": int(row["physical_factory_qubits"]),
        "memory_qubits": int(row["physical_memory_qubits"] or 0),
        "runtime_seconds": per_shot_seconds * SHOTS / (1 - float(row["error"])),
        "runtime_seconds_per_shot": per_shot_seconds,
        "error_budget": PER_SHOT_ERROR_BUDGET,
        "achieved_error": float(row["error"]),
    }

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "unknown"

results_record = {
    "schema_version": "1.0",
    "claim_id": CLAIM_ID,
    "reproduction_id": REPRODUCTION_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "producer": {"entrypoint": "notebook.ipynb", "cell_tag": "results-export"},
    "tooling": {
        "python": platform.python_version(),
        "packages": {name: package_version(name) for name in ("qdk", "numpy", "pandas")},
    },
    "inputs": {
        "factoring": {
            "kind": "rsa",
            "modulus_bits": N_BITS,
            "algorithm": "approx_modexp of Gidney (2025), Appendix A.1, built as one Q# circuit on the RSA-2048 challenge number",
            "parameters": {
                "s": S,
                "prime_bit_length": L,
                "window_w1": W1,
                "window_w3": W3,
                "window_w4": W4,
                "accumulator_length_f": F,
                "input_qubits_m": M,
                "mask_bits": PARAMS.mask_bits,
                "generator": PARAMS.generator,
                "num_primes": NUM_PRIMES,
                "residue_system_deviation_log2": RNS["L_mod_N_deviation_log2"],
                "lookup_table_digest_sha256": SCAN.table_digest,
                "zero_mask_pairs": int(sum(int(v.sum()) for v in SCAN.zero_pairs.values())),
                "deviant_shot_rate": P_DEVIANT,
                "deviant_shot_rate_percent": round(P_DEVIANT * 100, 3),
                "deviant_shot_rate_actual_wraparound": BOUND_ACTUAL_L["p_deviant"],
                "sampled_exponents": len(DEVIATIONS),
                "sampled_deviation_max": float(DEVIATIONS.max()),
                "sampled_deviation_mean": float(DEVIATIONS.mean()),
                "deviation_bound_epsilon": BOUND["epsilon"],
                "additions_per_shot": OPS_PER_SHOT["additions"],
                "lookups_per_shot": OPS_PER_SHOT["lookups"],
                "phaseups_per_shot": OPS_PER_SHOT["phaseups"],
                "lookup_circuit": "Appendix A.4 power-product lookup",
                "phaseup_circuit": "Appendix A.3 power-product phaseup",
                "addition_circuit": "Q# RippleCarryCGIncByLE",
                "verified_instances": len(VERIFICATION),
            },
        }
    },
    "logical": {
        "num_qubits": LOGICAL_QUBITS,
        "ccz_count": EXPECTED_TOFFOLIS,
        "ccix_count": 0,
        "toffoli_count": EXPECTED_TOFFOLIS,
        "toffoli_count_per_shot": TOFFOLIS_PER_SHOT,
        "shots_expected": SHOTS,
    },
    "physical": [
        layout_record(),
        qre_record("qre-yoked-cultivation", scenarios["qre-yoked-cultivation"],
                   "input register held in surface codes yoked by 2D parity checks"),
        qre_record("qre-hot-storage-only", scenarios["qre-hot-storage-only"],
                   "every qubit in ordinary surface-code patches"),
    ],
}

with open("results.json", "w", encoding="utf-8") as handle:
    json.dump(results_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")

print(json.dumps(results_record["logical"], indent=2))
for s in results_record["physical"]:
    print(s["scenario_id"], s["physical_qubits"], f"{s['runtime_seconds'] / 86400:.2f} days", f"d={s['code']['distance']}")

{
  "num_qubits": 1436,
  "ccz_count": 6515767508,
  "ccix_count": 0,
  "toffoli_count": 6515767508,
  "toffoli_count_per_shot": 714923280,
  "shots_expected": 9.113939481204216
}
paper-layout-rebuilt 931664 4.31 days d=25
qre-yoked-cultivation 734185 5.46 days d=23
qre-hot-storage-only 3744413 5.92 days d=25
